Collections
Unique record IDs
Dense vectors
Sparse vectors
Multiple named vectors
JSON payloads
Metadata indexes
Insert/update/upsert/delete
Filtering
Persistence
Snapshots
Replication and sharding
HTTP/gRPC server

Qdrant Database
│
├── Collection: company-documents
│     │
│     ├── Point 1
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     ├── Point 2
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     └── Point 3
│
├── Vector index
│     └── HNSW / related retrieval indexes
│
├── Payload index
│     └── category, source, page, user_id...
│
└── Storage
      ├── Memory
      └── Disk

In [ ]:
from __future__ import annotations

from pathlib import Path
from uuid import uuid4

from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models


PDF_PATH = Path("company_policy.pdf")
QDRANT_PATH = "./pdf_qdrant_data"
COLLECTION_NAME = "company-policy-rag"


if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH.resolve()}"
    )


# 1. Load PDF
loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print("PDF pages loaded:", len(pages))


# 2. Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

chunks = text_splitter.split_documents(pages)

print("Chunks created:", len(chunks))


# 3. Add useful metadata
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = PDF_PATH.name


# 4. Embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True
    },
)

dimension = len(
    embeddings.embed_query("dimension check")
)


# 5. Local Qdrant
client = QdrantClient(
    path=QDRANT_PATH
)


# 6. Create collection
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )


# 7. LangChain Qdrant vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


# 8. Use deterministic or stable IDs in production
chunk_ids = [
    str(uuid4())
    for _ in chunks
]


# 9. Add documents
inserted_ids = vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("Inserted chunks:", len(inserted_ids))

In [ ]:
query = "What is the annual leave policy?"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

context_documents = retriever.invoke(
    "What benefits are available to employees?"
)

context = "\n\n".join(
    document.page_content
    for document in context_documents
)

print(context)